# Usage of GC.Analysis.API for GC Analysis

In [1]:
#r "nuget: Microsoft.Diagnostics.Tracing.TraceEvent, 3.1.9"
#r "nuget: XPlot.Plotly"
#r "nuget: XPlot.Plotly.Interactive"
#r "nuget: Microsoft.Data.Analysis"
#r "nuget: Newtonsoft.Json"

using Etlx = Microsoft.Diagnostics.Tracing.Etlx;
using Microsoft.Data.Analysis;
using Microsoft.Diagnostics.Tracing.Analysis.GC;
using Microsoft.Diagnostics.Tracing.Analysis;
using Microsoft.Diagnostics.Tracing.Parsers.Clr;
using Microsoft.Diagnostics.Tracing;
using XPlot.Plotly;

using System.IO;
using Newtonsoft.Json;

Installed Packages Microsoft.Data.Analysis, 0.21.1 Microsoft.Diagnostics.Tracing.TraceEvent, 3.1.9 Newtonsoft.Json, 13.0.3 XPlot.Plotly, 4.0.6 XPlot.Plotly.Interactive, 4.0.7

Loading extensions from `Q:\.tools\.nuget\packages\microsoft.data.analysis\0.21.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

Loading extensions from `Q:\.tools\.nuget\packages\xplot.plotly.interactive\4.0.7\lib\net7.0\XPlot.Plotly.Interactive.dll`

Configuring PowerShell Kernel for XPlot.Plotly integration.

Installed support for XPlot.Plotly.

## Building and Using The GC Analysis API

In [2]:
dotnet build -c Release "..\..\GC.Analysis.API"

Switch: ..\..\GC.Analysis.API


In [3]:
#r "C:\Users\musharm\source\repos\performance4\artifacts\bin\GC.Analysis.API\Release\net8.0\GC.Analysis.API.dll"

using GC.Analysis.API;

#### Get All Processes From a Trace

In [ ]:
string path = @"C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\DATAS_Results\CitrineWin_normal\datas_citrine_normal_0\FortunesEf_Windows.gc.etlx";
path = @"C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\DATAS_Results\CitrineWin_normal\datas_citrine_normal_0\ConnectionCloseHttpSys_Windows.gc.etlx";
Analyzer gcTraceData = new Analyzer(tracePath: path); 
var benchmarks = gcTraceData.GetProcessGCData("Benchmarks")[0];

In [5]:
(benchmarks.GCs[0].PauseStartRelativeMSec, benchmarks.GCs[0].PauseDurationMSec + benchmarks.GCs[0].PauseStartRelativeMSec).Display()

Item1,7460.8211
Item2,7466.4346


In [65]:
public class HeapCountChangeSuspensionDetails_
{
    private sealed class SuspensionInfo
    {
        public double StartTimestampMSec { get; set; }
        public double EndTimestampMSec { get; set; } = double.NaN;
        public double Duration => EndTimestampMSec - StartTimestampMSec; 
        public GCInfo AssociatedGC { get; set; }
    }

    private sealed class GCInfo 
    {
        public double StartTime;
        public long Number;
        public GCType Type;
        public bool IsBackground => Type == GCType.BackgroundGC;
    }

    public HeapCountChangeSuspensionDetails_(GCProcessData gcProcessData, bool debug = false)
    {
        List<SuspensionInfo> suspensions = new();
        GCInfo? currentGC = null;
        GCInfo? previousGC = null;
        int currentBGCNumber = -1;

        var eventSource = gcProcessData.Parent.TraceLog.Events.GetSource();
        int processId = gcProcessData.ProcessID;

        // Only keep track of BGCs and GCs not contained within BGCs. 
        eventSource.Clr.GCStart += (data) =>
        {
            if (data.ProcessID != processId) return;

            GCInfo gc = new GCInfo { StartTime = data.TimeStampRelativeMSec, Type = data.Type, Number = data.Count };

            // If we aren't within a BGC, then check if the it's the start of BGC, if so, update the currentBGCNumber.
            if (currentBGCNumber == -1)
            {
                if (data.Type == GCType.BackgroundGC)
                {
                    currentBGCNumber = data.Count;
                }

                // Set the currentGC only in situations where we are not in BGC and the GC is not a BGC.
                // This is to keep track of the GCs that are not within a BGC (or a BGC).
                currentGC = gc;
            }
        };
        eventSource.Clr.GCStop += (data) =>
        {
            if (data.ProcessID != processId) return;

            // If we are at the end of a BGC, reset the currentBGCNumber.
            if (data.Count == currentBGCNumber)
            {
                currentBGCNumber = -1;
            }

            // If we are either at the end of a BGC or the GC is not enclosed in a BGC, set the previousGC to the currentGC.
            // Uninitialize the currentGC.
            if (currentBGCNumber == -1)
            {
                previousGC = new GCInfo { StartTime = currentGC.StartTime, Type = currentGC.Type, Number = currentGC.Number };
                currentGC = null; // At the end of a cycle => Reset the currentGC.
            }
        };

        eventSource.Clr.GCSuspendEEStart += (data) =>
        {
            if (data.ProcessID != processId) return;
            // Don't consider any other suspension reasons but those for `SuspendForGCPrep`.
            if (data.Reason != GCSuspendEEReason.SuspendForGCPrep) return;
            // Don't consider any events within a BGC.
            if (currentBGCNumber != -1) return;

            // Association is done with the previous GC since these events are fired _after_ a GC completes.
            suspensions.Add(new SuspensionInfo { StartTimestampMSec = data.TimeStampRelativeMSec, AssociatedGC = previousGC });
        };
        eventSource.Clr.GCRestartEEStop += (data) =>
        {
            if (data.ProcessID != processId) return;
            // Don't consider any events within a BGC.
            if (currentBGCNumber != -1) return;
            // In a situation where we encounter a RestartEEStop before a SuspendEEStart at the beginning of a trace.
            if (suspensions.Count == 0) return;

            // Expectation: GCSuspendEEStart is called before GCRestartEEStop => The association of the GC should have been done to the previous GC.
            // Check if the RestartEEStop is for the same GC as the last SuspendEEStart.
            // This is to handle the case where the Restart is not associated with a SuspendForGCPrep.
            if ((previousGC?.Number != suspensions[^1].AssociatedGC.Number)) return;

            suspensions[suspensions.Count - 1].EndTimestampMSec = data.TimeStampRelativeMSec;
        };
        eventSource.Process();

        HeapCountSuspensionData = new Dictionary<long, double>();
        foreach (var sus in suspensions)
        {
            if (!double.IsNaN(sus.Duration))
            {
                HeapCountSuspensionData[sus.AssociatedGC.Number] = sus.Duration;
            }

            else
            {
                Console.WriteLine($"Suspension without a corresponding restart for: {sus.AssociatedGC.Number} -- ignoring.");
            }
        }
    }

    public Dictionary<long, double> HeapCountSuspensionData { get; }
}

In [66]:
HeapCountChangeSuspensionDetails_ suspensionDetails = new(benchmarks, true);
suspensionDetails.HeapCountSuspensionData.Display()

/* From algo below:
key	value
4 0.23109999999905995
14 0.3474999999998545
35 0.3567000000002736
105 0.5146999999997206
*/

key,value
4,0.23109999999905995
14,0.3474999999998545
35,0.3567000000002736
105,0.5146999999997206


In [6]:
public class HeapCountChangeSuspensionDetails
{
    private record SuspensionInfo(double timestamp, string reason);
    private sealed class GCStartEndInfo
    {
        public double StartTime;
        public double EndTime;
        public long Count;
        public GCType Type;
    }

    public HeapCountChangeSuspensionDetails(GCProcessData gcProcessData, bool debug = false)
    {
        List<SuspensionInfo> suspensions = new();
        SortedDictionary<int, GCStartEndInfo> gcData = new();

        var eventSource = gcProcessData.Parent.TraceLog.Events.GetSource();
        int processId = gcProcessData.ProcessID;

        eventSource.Clr.GCStart += (data) =>
        {
            if (data.ProcessID != processId) return;
            int gcCount = data.Count;
            if (!gcData.TryGetValue(gcCount, out var gcInfo))
            {
                gcInfo = new GCStartEndInfo();
                gcInfo.StartTime = data.TimeStampRelativeMSec;
                gcInfo.Count = gcCount;
                gcInfo.Type = data.Type;
                gcData[gcCount] = gcInfo;
            }
        };
        eventSource.Clr.GCStop += (data) =>
        {
            if (data.ProcessID != processId) return;
            int gcCount = data.Count;
            if (!gcData.TryGetValue(gcCount, out var gcInfo))
            {
                throw new Exception($"GCStop event without GCStart event for GC {gcCount}");
            }

            else
            {
                gcInfo.EndTime = data.TimeStampRelativeMSec;
            }
        };

        eventSource.Clr.GCSuspendEEStart += (data) =>
        {
            if (data.ProcessID != processId) return;
            suspensions.Add(new SuspensionInfo(data.TimeStampRelativeMSec, data.Reason.ToString()));
        };
        eventSource.Clr.GCRestartEEStop += (data) =>
        {
            if (data.ProcessID != processId) return;
            suspensions.Add(new SuspensionInfo(data.TimeStampRelativeMSec, /* Placeholder not considered */ "RestartEEStop"));
        };
        eventSource.Process();

        // 1. Gather all the pairs of suspension starts and stops with SuspendForGCPrep reason.
        List<(SuspensionInfo start, SuspensionInfo stop)> suspendedForGCPrepPairs = new();
        for (int i = 0; i < suspensions.Count - 2; i += 2)
        {
            if (suspensions[i + 1].reason != "RestartEEStop")
            {
                throw new Exception($"Suspension not ended properly for {suspensions[i + 1].timestamp}");
            }

            if (suspensions[i].reason != GCSuspendEEReason.SuspendForGCPrep.ToString()) continue;
            suspendedForGCPrepPairs.Add((suspensions[i], suspensions[i + 1]));
        }

        if (debug)
        {
            Console.WriteLine("Suspensions for GC Prep Before Removing BGCs:");
            suspendedForGCPrepPairs.Display();
        }

        // 2. Remove the suspensions encapsulated in a BGC.
        foreach (var gc in gcData)
        {
            if (gc.Value.Type == GCType.BackgroundGC)
            {
                suspendedForGCPrepPairs.RemoveAll(s => s.start.timestamp >= gc.Value.StartTime && s.stop.timestamp <= gc.Value.EndTime);
            }
        }

        if (debug)
        {
            Console.WriteLine("Suspensions for GC Prep After Removing BGCs:");
            suspendedForGCPrepPairs.Display();
        }

        // 3. Associate the suspension with a GC.
        Dictionary<long, double> data = new();
        if (suspendedForGCPrepPairs.Count > 0)
        {
            List<GCStartEndInfo> gcDataAsList = gcData.Select(x => x.Value).ToList();
            for (int gcIdx = 0; gcIdx < gcData.Count; gcIdx++)
            {
                GCStartEndInfo gc = gcDataAsList[gcIdx];
                if (gc.Type == GCType.BackgroundGC) continue;

                GCStartEndInfo nextGC = (gcIdx + 1) < gcData.Count ? gcDataAsList[gcIdx + 1] : default;
                double suspensionDuration = double.NaN;

                foreach (var suspension in suspendedForGCPrepPairs)
                {
                    if ((suspension.start.timestamp > gc.StartTime && suspension.stop.timestamp < nextGC?.EndTime) || 
                        (suspension.start.timestamp > gc.StartTime && nextGC == default))
                    {
                        suspensionDuration = suspension.stop.timestamp - suspension.start.timestamp;
                        data[gc.Count] = suspensionDuration;
                    }
                }
            }
        }

        HeapCountSuspensionData = data;
    }

    public Dictionary<long, double> HeapCountSuspensionData { get; }
}

In [7]:
HeapCountChangeSuspensionDetails suspensionDetails = new(benchmarks, true);
suspensionDetails.HeapCountSuspensionData.Display()

Suspensions for GC Prep Before Removing BGCs:


index value 0 (SuspensionInfo { timestamp = 8494.6774, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 8494.9085, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 8494.6774, reason = SuspendForGCPrep } timestamp 8494.6774 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 8494.9085, reason = RestartEEStop } timestamp 8494.9085 reason RestartEEStop 1 (SuspensionInfo { timestamp = 8505.2604, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 8508.2883, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 8505.2604, reason = SuspendForGCPrep } timestamp 8505.2604 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 8508.2883, reason = RestartEEStop } timestamp 8508.2883 reason RestartEEStop 2 (SuspensionInfo { timestamp = 8859.7729, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 8860.1204, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 8859.7729, reason = SuspendForGCPrep } timestamp 8859.7729 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 8860.1204, reason = RestartEEStop } timestamp 8860.1204 reason RestartEEStop 3 (SuspensionInfo { timestamp = 9593.996, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 9594.3527, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 9593.996, reason = SuspendForGCPrep } timestamp 9593.996 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 9594.3527, reason = RestartEEStop } timestamp 9594.3527 reason RestartEEStop 4 (SuspensionInfo { timestamp = 9630.2137, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 9633.278, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 9630.2137, reason = SuspendForGCPrep } timestamp 9630.2137 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 9633.278, reason = RestartEEStop } timestamp 9633.278 reason RestartEEStop 5 (SuspensionInfo { timestamp = 13327.6183, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 13328.133, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 13327.6183, reason = SuspendForGCPrep } timestamp 13327.6183 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 13328.133, reason = RestartEEStop } timestamp 13328.133 reason RestartEEStop

Suspensions for GC Prep After Removing BGCs:


index value 0 (SuspensionInfo { timestamp = 8494.6774, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 8494.9085, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 8494.6774, reason = SuspendForGCPrep } timestamp 8494.6774 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 8494.9085, reason = RestartEEStop } timestamp 8494.9085 reason RestartEEStop 1 (SuspensionInfo { timestamp = 8859.7729, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 8860.1204, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 8859.7729, reason = SuspendForGCPrep } timestamp 8859.7729 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 8860.1204, reason = RestartEEStop } timestamp 8860.1204 reason RestartEEStop 2 (SuspensionInfo { timestamp = 9593.996, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 9594.3527, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 9593.996, reason = SuspendForGCPrep } timestamp 9593.996 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 9594.3527, reason = RestartEEStop } timestamp 9594.3527 reason RestartEEStop 3 (SuspensionInfo { timestamp = 13327.6183, reason = SuspendForGCPrep }, SuspensionInfo { timestamp = 13328.133, reason = RestartEEStop }) Item1 SuspensionInfo { timestamp = 13327.6183, reason = SuspendForGCPrep } timestamp 13327.6183 reason SuspendForGCPrep Item2 SuspensionInfo { timestamp = 13328.133, reason = RestartEEStop } timestamp 13328.133 reason RestartEEStop

key,value
4,0.23109999999905995
14,0.3474999999998545
35,0.3567000000002736
105,0.5146999999997206


In [8]:
Console.WriteLine($"Current Process ID: {System.Diagnostics.Process.GetCurrentProcess().Id}");

#!about

Current Process ID: 32508


.NET Interactive© 2020 Microsoft CorporationVersion: 1.0.522904+cdfa48b2ea1a27dfe0f545c42a34fd3ec7119074Library version: 1.0.0-beta.24229.4+cdfa48b2ea1a27dfe0f545c42a34fd3ec7119074Build date: 2024-05-22T20:20:42.3691153Zhttps://github.com/dotnet/interactive
